<a href="https://colab.research.google.com/github/sebabecerra/Fondos-de-Pensiones-Codes/blob/main/hack_spensiones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [118]:
!rm -rf /content/carteras_agregadas/

!rm -rf /content/carteras_fp
!rm -rf /content/html_v2
!rm -rf /content/cartera_agregada_total.csv

!rm -rf /content/html_v2

!rm -rf /content/v2_csv
!rm -rf /content/csv_historico/

!rm -rf /content/csv_202512/
!rm -rf /content/html_202512/
!rm -rf /content/carteras_menu_v2
!rm -rf /content/carteras_202512/

In [117]:
import os
import re
import unicodedata
from io import StringIO
from urllib.parse import urljoin

import html as html_lib
import pandas as pd
import requests
from bs4 import BeautifulSoup


BASE_URL = "https://www.spensiones.cl"


def _decode_html(resp: requests.Response) -> str:
    """
    Decodifica HTML de forma robusta para evitar mojibake (INVERSION, ECONOMICO, N0, etc.).
    1) intenta UTF-8
    2) si falla, usa apparent_encoding
    3) fallback latin1
    """
    raw = resp.content

    # 1) UTF-8 (lo más probable aquí)
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        pass

    # 2) encoding estimado por requests/chardet
    enc = (resp.apparent_encoding or "").strip().lower()
    if enc:
        try:
            return raw.decode(enc, errors="replace")
        except Exception:
            pass

    # 3) fallback seguro
    return raw.decode("latin1", errors="replace")


def limpiar_nombre(texto: str, max_len: int = 180) -> str:
    """
    Normaliza un título para nombre de archivo:
    - decodifica entidades HTML (&oacute; etc.)
    - normaliza unicode (quita acentos)
    - deja solo [a-z0-9_ -]
    - colapsa underscores
    """
    texto = html_lib.unescape(texto).strip()

    # Normaliza y quita acentos
    texto = unicodedata.normalize("NFKD", texto)
    texto = texto.encode("ascii", "ignore").decode("ascii")

    # Limpieza a slug
    texto = re.sub(r"[^\w\s-]", "", texto)
    texto = re.sub(r"\s+", "_", texto)
    texto = re.sub(r"_+", "_", texto)

    return texto[:max_len].strip("_")


def descargar_html_y_csv(periodo: str, base_dir: str = "/content/carteras_agregadas") -> None:
    html_dir = os.path.join(base_dir, "html", periodo)
    csv_dir = os.path.join(base_dir, "csv", periodo)
    os.makedirs(html_dir, exist_ok=True)
    os.makedirs(csv_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Referer": f"{BASE_URL}/apps/centroEstadisticas/paginaCuadrosCCEE.php",
    })

    url_intermedia = (
        f"{BASE_URL}/apps/loadCarteras/loadCarAgr.php"
        f"?menu=sci&menuN1=estfinfp&menuN2=NOID"
        f"&orden=20&periodo={periodo}&ext=.php"
    )

    print("🔗 Página intermedia:")
    print(url_intermedia)

    r = session.get(url_intermedia, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    links = [
        urljoin(BASE_URL, a["href"])
        for a in soup.find_all("a", title="Html", href=True)
        if "genera_xsl_v2.0.php" in a["href"]
    ]

    print(f"\n📎 {len(links)} links encontrados")

    for i, link in enumerate(links, 1):
        print(f"\n⬇️ [{i}/{len(links)}] {link}")

        resp = session.get(link, timeout=60)
        resp.raise_for_status()

        # === HTML "base" (NO tocar con replace de números) ===
        html_base = _decode_html(resp)
        html_base = html_base.replace("\xa0", " ").replace("Â", "")

        soup_html = BeautifulSoup(html_base, "html.parser")

        # Título desde <h3> (desde html_base bien decodificado)
        h3 = soup_html.find("h3")
        titulo = h3.get_text(" ", strip=True) if h3 else f"CUADRO_{i:02d}"

        nombre_base = limpiar_nombre(titulo).lower()  # ✅ minúsculas
        if not nombre_base:
            nombre_base = f"cuadro_{i:02d}"

        # Guardar HTML (el base, limpio, sin tocar números globalmente)
        html_path = os.path.join(html_dir, f"{nombre_base}.html")
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(html_base)
        print(f"💾 HTML → {html_path}")

        # === HTML para tablas (ACÁ se deja tu corrección) ===
        html_tablas = html_base.replace(".", "").replace(",", ".")  # ✅ se deja

        # HTML → CSV (primera tabla)
        try:
            tablas = pd.read_html(StringIO(html_tablas))
        except ValueError:
            print("  ⚠️ Sin tablas → CSV no generado")
            continue

        if not tablas:
            print("  ⚠️ HTML sin tablas → CSV no generado")
            continue

        df = tablas[0]
        csv_path = os.path.join(csv_dir, f"{nombre_base}.csv")
        df.to_csv(csv_path, index=False)
        print(f"📄 CSV  → {csv_path}")


# ▶ EJECUCIÓN
descargar_html_y_csv(
    periodo="202512",
    base_dir="/content/carteras_agregadas"
)


🔗 Página intermedia:
https://www.spensiones.cl/apps/loadCarteras/loadCarAgr.php?menu=sci&menuN1=estfinfp&menuN2=NOID&orden=20&periodo=202512&ext=.php

📎 70 links encontrados

⬇️ [1/70] https://www.spensiones.cl/apps/carteras/genera_xsl_v2.0.php?param=Ly84eVVNR3hvQU8yVU1McFB1dnlyU0tmdGgwdnpseEF3WlhQY295YnRJb20xei9BK1VmbkVBPT0=
💾 HTML → /content/carteras_agregadas/html/202512/cuadro_no_1_cartera_agregada_de_los_fondos_de_pensiones_por_tipo_de_fondo.html
📄 CSV  → /content/carteras_agregadas/csv/202512/cuadro_no_1_cartera_agregada_de_los_fondos_de_pensiones_por_tipo_de_fondo.csv

⬇️ [2/70] https://www.spensiones.cl/apps/carteras/genera_xsl_v2.0.php?param=Ly84eVVNR3hvQU8yVU1McFB1dnlyZWJ6dmpQdDMyYnp3WlhQY295YnRJb20xei9BK1VmbkVBPT0=
💾 HTML → /content/carteras_agregadas/html/202512/cuadro_no_2_cartera_agregada_de_los_fondos_de_pensiones_tipo_a_por_afp.html
📄 CSV  → /content/carteras_agregadas/csv/202512/cuadro_no_2_cartera_agregada_de_los_fondos_de_pensiones_tipo_a_por_afp.csv

⬇️ [3/70] https